# Classical Text Classification

---

<img src="https://www.dropbox.com/scl/fi/b1vbv4c4m5vikt6s08n62/nlp.png?rlkey=r5t9i1socnr84jk2slvx2pylw&raw=1" align="center"/>

## The Scenario

You are a data scientist. You just inherited a folder of **10,000 Yelp reviews** with star ratings. Your manager wants to know, by tomorrow, whether a given review is **positive** (4-5 stars) or **negative** (1-2 stars).

You have:
- No GPU.
- No labeled "sentiment" column (only star ratings).
- One afternoon.
- A laptop with `scikit-learn` installed.

Deep learning is overkill. What you need is a **strong classical baseline** that a future neural network will have to beat. That is what we will build today.

## Learning Objectives

By the end of this notebook you will be able to:
1. Frame a text problem as a supervised learning task.
2. Convert raw text into numeric features with `CountVectorizer` (Bag of Words) and `TfidfVectorizer` (TF-IDF).
3. Use n-grams and stop words, and know when each helps.
4. Train and evaluate **Multinomial Naive Bayes**, **Random Forest**, and **Logistic Regression** on text.
5. Read a `classification_report` critically (precision, recall, F1) and compare models with confusion matrices.
6. Engineer hand-crafted features (review length, punctuation, sentiment) and combine them with a TF-IDF matrix via `scipy.sparse.hstack`.
7. Apply the full pipeline to a brand-new dataset (SMS spam) to prove the workflow generalizes.

## Prerequisites
- Notebook 1 (preprocessing basics: tokenization, stop words, stemming).
- Comfort with `pandas` and `numpy`.

## The Golden Rule of Classical ML for Text

> **Always compute the majority-class baseline first.** If 82% of your reviews are 5-star, a model that always predicts "5 stars" gets 82% accuracy. Any real model must beat that bar. We will enforce this discipline throughout.



## Section 0: Environment Setup

Run the next cell once per Colab session. If you are on a local machine with `requirements.txt` already installed, you can skip it.

In [ ]:
# Install the small set of libraries we need for classical text classification.
# - scikit-learn: vectorizers, classifiers, metrics
# - textblob:      quick rule-based sentiment as a hand-crafted feature
# - spacy:         loaded here for consistency with Notebook 1 (we barely use it)
# - pandas/numpy:  data wrangling
# - matplotlib/seaborn: plots (confusion matrix heatmaps)
!pip install --quiet --upgrade scikit-learn textblob spacy pandas numpy matplotlib seaborn scipy

### The Classifier Pipeline

Every classical text classifier we will build today follows the same three-step pipeline:

```
  raw text  ->  vectorizer  ->  classifier  ->  prediction
 ("I loved     (CountVec or   (MultinomialNB,    (positive /
   this!")     TfidfVec)       RandomForest,      negative)
                                LogReg, ...)
```

The vectorizer turns strings into a sparse numeric matrix (rows = documents, columns = words). The classifier is a standard scikit-learn estimator that knows nothing special about text. This separation is why classical NLP is so powerful: **you can swap vectorizers and classifiers independently.**

In [ ]:
# Core imports -- grouped by purpose (visualization -> data -> model -> metrics)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from textblob import TextBlob

# ----- Hyperparameters (single source of truth) ----------------------------------
SEED            = 42
TEST_SIZE       = 0.2
MAX_FEATURES    = 20000    # Cap the vocabulary so the matrix stays tractable
NGRAM_RANGE     = (1, 2)   # Unigrams + bigrams
MIN_DF          = 2        # Ignore words that appear in < 2 documents
MAX_DF          = 0.95     # Ignore words that appear in > 95% of documents
RF_N_ESTIMATORS = 100

np.random.seed(SEED)

sns.set_theme(style='whitegrid')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')
print('Environment setup complete.')

## Section 1: Load Yelp & Frame the Problem

We download the Yelp reviews CSV straight from a public Dropbox URL (the `dl=1` query flag forces a direct download, unlike `dl=0` which returns a preview page).

**Binary filter.** The original dataset has 1-5 stars. We keep **only 1-star and 5-star reviews**, then relabel them as a binary target:

```python
y = (stars >= 4).astype(int)   # 1 = positive, 0 = negative
```

Why binary? It turns an ambiguous ordinal problem into a crisp classification problem -- exactly the kind of scoping move a data scientist makes on day one.

In [ ]:
# Direct-download URL (note dl=1 at the end).
YELP_URL = 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(YELP_URL)

print(f'Full dataset: {len(yelp):,} rows')
print('\nStar distribution:')
print(yelp['stars'].value_counts().sort_index())

# Keep only the extremes -- clearest positive/negative signal.
yelp_bw = yelp[yelp['stars'].isin([1, 5])].reset_index(drop=True)
print(f'\nAfter 1/5-star filter: {len(yelp_bw):,} rows')

X = yelp_bw['text']
y = (yelp_bw['stars'] >= 4).astype(int)   # 1 = positive, 0 = negative

# Stratified split preserves the class ratio in both train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print(f'\nTrain: {len(X_train):,}   Test: {len(X_test):,}')
print(f'Positive rate (train): {y_train.mean():.3f}')

## Section 2: Bag of Words

The simplest way to turn text into numbers: **count how often each word appears in each document.** The resulting matrix has one row per document and one column per unique word in the vocabulary. Order is lost -- hence "bag of words".

Toy example with 3 documents:
```
doc 0: "I love pizza"
doc 1: "pizza is great"
doc 2: "I hate pizza"

vocabulary: {great, hate, is, love, pizza}   (the, I are dropped as single letters or repeats)

             great  hate  is  love  pizza
doc 0          0     0    0    1      1
doc 1          1     0    1    0      1
doc 2          0     1    0    0      1
```

In scikit-learn this is one line: `CountVectorizer().fit_transform(corpus)`.

In [ ]:
# Demo: the toy example above, in code
toy = ["I love pizza", "pizza is great", "I hate pizza"]

demo_vect = CountVectorizer()
demo_dtm  = demo_vect.fit_transform(toy)

print('Vocabulary:', demo_vect.vocabulary_)
print('\nDocument-term matrix:')
print(pd.DataFrame(demo_dtm.toarray(),
                   columns=demo_vect.get_feature_names_out(),
                   index=[f'doc {i}' for i in range(3)]))

In [ ]:
# Real Yelp data -- fit CountVectorizer on the training set only.
# Fitting on the TRAIN set and then transforming the TEST set is essential.
# (Fitting on the test set leaks test-time vocabulary into training.)
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)

print(f'X_train_bow shape: {X_train_bow.shape}   (documents, vocab)')
print(f'Vocabulary size:   {len(bow.vocabulary_):,}')
print(f'Non-zero entries:  {X_train_bow.nnz:,}')
print(f'Density:           {X_train_bow.nnz / np.prod(X_train_bow.shape):.4%}')

### Sparse Matrices -- Why We Care

Notice the density printed above: about **0.5%**. That is, 99.5% of the cells in the document-term matrix are zero. Storing them all as a dense `numpy` array would waste memory:

```python
# Back-of-envelope
n_docs, n_vocab = X_train_bow.shape       # e.g. 3200 x 17000
dense_bytes = n_docs * n_vocab * 8        # float64
# roughly 430 MB for this tiny corpus!
```

scikit-learn returns a **`scipy.sparse.csr_matrix`** (Compressed Sparse Row). Only non-zero values are stored. Every classifier in this notebook accepts sparse input directly -- so only call `.toarray()` on small slices for inspection, never on the whole matrix.

### The Majority-Class Baseline

Before we train any classifier, compute the number our model must beat:

In [ ]:
# What accuracy does a trivial model that always predicts the majority class get?
majority_class = y_train.mode()[0]
baseline_preds = np.full(len(y_test), majority_class)
baseline_acc   = accuracy_score(y_test, baseline_preds)

print(f'Majority class in train: {majority_class}  (1 = positive)')
print(f'Baseline accuracy:       {baseline_acc:.4f}')
print('\nAny real model must beat this, or we should not ship it.')

### Lab 2.1 -- Vocabulary Size vs. Accuracy

The `max_features` argument of `CountVectorizer` caps the vocabulary to the top-N most frequent terms. Does a bigger vocabulary always help? Test three sizes and plot the downstream accuracy of a `MultinomialNB` classifier.

**Your tasks:**
1. For each `n` in `[1000, 5000, 20000]`:
   - Fit a `CountVectorizer(max_features=n)` on `X_train`.
   - Transform `X_train` and `X_test`.
   - Train a `MultinomialNB`.
   - Record the test accuracy.
2. Plot `max_features` on x-axis (log scale), accuracy on y-axis.
3. Print the best `max_features` and its accuracy.

Expected: accuracy climbs then plateaus around 5000-20000.


In [ ]:
# Lab 2.1 -- starter code
feature_sizes = [1000, 5000, 20000]
accuracies    = []

for n in feature_sizes:
    # 1. Build a CountVectorizer with max_features=n
    vect = None  # YOUR CODE

    # 2. Fit on X_train, transform X_train and X_test
    Xtr = None  # YOUR CODE
    Xte = None  # YOUR CODE

    # 3. Train a MultinomialNB on (Xtr, y_train)
    clf = None  # YOUR CODE

    # 4. Compute test accuracy and append to `accuracies`
    acc = None  # YOUR CODE
    accuracies.append(acc)
    print(f'max_features={n:>6}   accuracy={acc}')

# 5. Plot max_features vs. accuracy (log-x)
if all(a is not None for a in accuracies):
    plt.figure(figsize=(7, 4))
    plt.semilogx(feature_sizes, accuracies, marker='o')
    plt.xlabel('max_features')
    plt.ylabel('test accuracy')
    plt.title('Vocabulary size vs. Naive Bayes accuracy')
    plt.grid(True, which='both', alpha=0.3)
    plt.show()

## Section 3: Stop Words & N-grams

### Stop words

Common words like `the`, `is`, `a`, `of` appear in nearly every document, so they carry little signal for most tasks. Dropping them typically shrinks the vocabulary **and** reduces noise. scikit-learn ships a built-in English list: `stop_words='english'`.

**Warning:** In sentiment analysis stop-word removal can *hurt*. Consider

```
"this movie is not good"   ->  ["movie", "good"]    (without stop words)
```

The word `not` is in the default stop list. Remove it and the review looks positive. Rule of thumb: **default stop lists are a starting point, never a finishing point.**

### N-grams

A unigram is one word (`good`). A bigram is two consecutive words (`not good`). With `ngram_range=(1, 2)` the vectorizer produces both. Bigrams capture local word order that bag-of-words loses. Trade-off: vocabulary explodes, and many bigrams appear only once (noise).

In [ ]:
# Compare three vectorizer configurations on the same NB classifier.
configs = {
    'unigrams, no stop words':  CountVectorizer(ngram_range=(1, 1)),
    'unigrams + english stops': CountVectorizer(ngram_range=(1, 1), stop_words='english'),
    'unigrams + bigrams':       CountVectorizer(ngram_range=(1, 2), min_df=MIN_DF),
}

for name, v in configs.items():
    Xtr = v.fit_transform(X_train)
    Xte = v.transform(X_test)
    nb  = MultinomialNB().fit(Xtr, y_train)
    acc = accuracy_score(y_test, nb.predict(Xte))
    print(f'{name:30}  vocab={Xtr.shape[1]:>6}   acc={acc:.4f}')

In [ ]:
# Peek at some bigrams -- reassuring that sentiment-bearing pairs show up.
v = CountVectorizer(ngram_range=(2, 2), min_df=5)
v.fit(X_train)
bigrams = [b for b in v.get_feature_names_out()
           if 'not' in b.split() or 'very' in b.split()]
print('A few negation/intensity bigrams in the corpus:')
print(bigrams[:15])

### Lab 3.1 -- Custom Domain Stop Words

On a restaurant review corpus, words like `restaurant`, `food`, and `place` appear in almost every document -- they are essentially domain-specific stop words. Let's confirm that removing them does **not** hurt accuracy (and may even help a tiny bit).

**Your tasks:**
1. Create a list `my_stops` that extends sklearn's English list with restaurant-specific terms: `['restaurant', 'food', 'place', 'time', 'order']`.
2. Fit a `CountVectorizer` with `stop_words=my_stops` and `ngram_range=(1, 2)`.
3. Train a Naive Bayes, compare accuracy to the plain `stop_words='english'` version above.

Hint: `from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS`, then `list(ENGLISH_STOP_WORDS) + extras`.


In [ ]:
# Lab 3.1 -- starter code
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

extra_stops = ['restaurant', 'food', 'place', 'time', 'order']

# 1. Build the combined stop-word list
my_stops = None  # YOUR CODE

# 2. Vectorize with stop_words=my_stops and ngram_range=(1, 2)
vect_custom = None  # YOUR CODE
Xtr = None  # YOUR CODE
Xte = None  # YOUR CODE

# 3. Train MultinomialNB and score
nb_custom = None  # YOUR CODE
acc_custom = None  # YOUR CODE

if acc_custom is not None:
    print(f'Custom domain stop words accuracy: {acc_custom:.4f}')

## Section 4: TF-IDF

Bag-of-words treats every word count equally. But a word that appears in **every** document (like `the`) should count for less than a word that appears in only a few.

**TF-IDF** = Term Frequency x Inverse Document Frequency.

For a term `t` in document `d` out of `N` total documents where `df(t)` is the number of documents containing `t`:

```
tf(t, d)  = count of t in d  (sometimes normalized by |d|)
idf(t)    = log( N / df(t) ) + 1
tfidf     = tf * idf
```

Intuition:
- `idf` is **high** for rare words (they are informative).
- `idf` is **near zero** for words that appear in nearly every document.
- Multiplying by `tf` means a rare word that also appears many times in this document gets a big score.

When to prefer **BoW** over **TF-IDF**: very short texts (tweets, headlines) where every word is rare anyway, or when you need exact counts for Naive Bayes.

In [ ]:
# Inspect the .idf_ array -- what words get the highest scores?
tfidf_demo = TfidfVectorizer(min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES)
tfidf_demo.fit(X_train)

idf_series = pd.Series(tfidf_demo.idf_, index=tfidf_demo.get_feature_names_out())

print('Lowest IDF (appear almost everywhere -- near-useless):')
print(idf_series.nsmallest(10).round(3))
print('\nHighest IDF (rare, potentially informative):')
print(idf_series.nlargest(10).round(3))

In [ ]:
# Train Multinomial NB on TF-IDF features with the hyperparameters from the top of the notebook.
tfidf = TfidfVectorizer(
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
    stop_words='english',
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

nb_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)
pred_tfidf = nb_tfidf.predict(X_test_tfidf)
print(f'NB + TF-IDF accuracy: {accuracy_score(y_test, pred_tfidf):.4f}')
print(f'Vocabulary size:      {X_train_tfidf.shape[1]:,}')

### Lab 4.1 -- Compute TF-IDF by Hand

Sanity-check your understanding by reproducing sklearn's output on a 3-document toy corpus.

Corpus:
```
doc 0: "call you tonight"
doc 1: "call me a cab"
doc 2: "please call me please"
```

**Your tasks:**
1. Build the vocabulary manually (sorted alphabetically, ignoring single-letter words like `a` which sklearn also drops by default).
2. Compute the `tf` matrix (counts) as a numpy array of shape `(3, V)`.
3. Compute `df` (number of documents each word appears in) as a length-`V` array.
4. Compute `idf = log(N / df) + 1` (using `np.log`, N=3).
5. Compute `tfidf = tf * idf` and **L2-normalize each row** (sklearn does this by default).
6. Verify your matrix matches `TfidfVectorizer().fit_transform(simple_train).toarray()`.


In [ ]:
# Lab 4.1 -- starter code
simple_train = ['call you tonight', 'call me a cab', 'please call me please']

# 1. Vocabulary (sklearn drops single-char tokens by default).
vocab = None  # YOUR CODE -- a sorted list of the 5 remaining words

# 2. Term-frequency matrix (3 x V)
tf = None  # YOUR CODE  (hint: loop over docs, count with str.split())

# 3. Document frequency (length V): how many docs each word appears in
df = None  # YOUR CODE  (hint: (tf > 0).sum(axis=0))

# 4. Inverse document frequency
N = 3
idf = None  # YOUR CODE  (hint: np.log(N / df) + 1)

# 5. Raw tfidf, then L2-normalize each row
tfidf_raw = None  # YOUR CODE
tfidf_manual = None  # YOUR CODE  (hint: divide each row by its L2 norm)

# 6. Compare with sklearn
tfidf_sklearn = TfidfVectorizer().fit_transform(simple_train).toarray()

if tfidf_manual is not None:
    print('Manual:')
    print(np.round(tfidf_manual, 3))
    print('\nsklearn:')
    print(np.round(tfidf_sklearn, 3))
    print('\nMatch?', np.allclose(tfidf_manual, tfidf_sklearn, atol=1e-3))

## Section 5: Classifiers -- NB, RF, Logistic Regression

### Why Multinomial Naive Bayes is *the* text baseline

MNB assumes each word is independent of every other word, given the class label. That assumption is wildly wrong (word order, dependencies, etc.) but surprisingly effective on text because:

1. **Fast.** Training is a single pass over the sparse matrix.
2. **Robust on small data.** Few parameters to estimate.
3. **Handles high-dimensional sparse input natively.**

It is the "always-start-here" choice for text classification. If a shinier model cannot beat MNB by a clear margin, think carefully before shipping the shinier model.

### The other two contenders
- **Random Forest** -- an ensemble of decision trees, each trained on a bootstrap sample with a random feature subset. Captures non-linear interactions. Memory-hungry on sparse text.
- **Logistic Regression** -- a linear model that outputs calibrated probabilities. The workhorse of classical NLP in industry.

In [ ]:
# Train all three on TF-IDF features, record predictions.
models = {
    'MultinomialNB'      : MultinomialNB(),
    'RandomForest'       : RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1),
    'LogisticRegression' : LogisticRegression(max_iter=1000, random_state=SEED),
}

predictions = {}
for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    predictions[name] = preds
    print(f'{name:22} accuracy = {accuracy_score(y_test, preds):.4f}')

print(f'\nMajority-class baseline = {baseline_acc:.4f}')

In [ ]:
# Classification reports side-by-side (precision, recall, F1, support)
for name, preds in predictions.items():
    print(f'=== {name} ===')
    print(classification_report(y_test, preds, target_names=['negative (1*)', 'positive (5*)']))

### Reading the `classification_report`

For each class:
- **precision = TP / (TP + FP)** -- "of the reviews I flagged positive, how many truly were?"
- **recall    = TP / (TP + FN)** -- "of the actual positives, how many did I catch?"
- **f1        = 2 * P * R / (P + R)** -- harmonic mean.
- **support**   = number of true examples of that class in the test set.

When to optimize which:
- **Spam filter** -> precision matters (a false positive sends a real email to junk).
- **Cancer screening** -> recall matters (a false negative misses a patient).
- **Balanced product classification** -> F1 or macro-F1.

In [ ]:
# Side-by-side confusion matrices with seaborn heatmaps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'], ax=ax, cbar=False)
    ax.set_title(f'{name}\nacc={accuracy_score(y_test, preds):.3f}')
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')
plt.tight_layout()
plt.show()

### Lab 5.1 -- Pick the Winner and Explain Why

Based on the three classification reports and the confusion matrices above:

**Your tasks:**
1. Fill in the dict `results` with the test accuracy of each of the three models.
2. Identify the winner (highest accuracy).
3. In a print statement, answer: **why** do you think the winner beat the others on this dataset? Consider the class balance, data size, and the fact that TF-IDF features are roughly linear signals.

No single right answer -- we want your reasoning, not just the number.


In [ ]:
# Lab 5.1 -- starter code
# 1. Fill this dict from the cells above
results = {
    'MultinomialNB'      : None,  # YOUR CODE -- accuracy as a float
    'RandomForest'       : None,  # YOUR CODE
    'LogisticRegression' : None,  # YOUR CODE
}

# 2. Find the winner
if all(v is not None for v in results.values()):
    winner = None  # YOUR CODE -- hint: max(results, key=results.get)
    print(f'Winner: {winner}  ({results[winner]:.4f})')

    # 3. Your reasoning -- replace this string
    reasoning = None  # YOUR CODE -- one or two sentences
    if reasoning:
        print(f'\nWhy: {reasoning}')

## Section 6: Sentiment Scores & Feature Engineering

A **sentiment score** is a number, typically in `[-1, +1]`, summarizing how positive or negative a piece of text is. We will use TextBlob's rule-based polarity score as a *feature*, not a classifier.

Polarity is just a number. Feeding it alongside TF-IDF lets the classifier combine "what words appear" with "how positive the review reads overall." We will also add cheap hand-crafted features:
- **review length** (in characters) -- angry 1-star reviews tend to be long.
- **exclamation count** -- `!` is a mild positive/emphasis signal.
- **uppercase word ratio** -- ALL CAPS usually = strong emotion.

All three are single numbers per document; we stack them onto the TF-IDF matrix with `scipy.sparse.hstack`.

In [ ]:
# Compute hand-crafted features for train and test.
def hand_features(texts):
    """Return an (n_docs, 4) dense numpy array of hand-crafted features."""
    feats = np.zeros((len(texts), 4), dtype=np.float64)
    for i, t in enumerate(texts):
        words = t.split()
        feats[i, 0] = len(t)                                          # char length
        feats[i, 1] = t.count('!')                                    # exclamation count
        feats[i, 2] = sum(w.isupper() for w in words) / max(len(words), 1)  # upper ratio
        feats[i, 3] = TextBlob(t).sentiment.polarity                  # polarity [-1, 1]
    return feats

print('Computing hand-crafted features on train... (~30-60s)')
train_hand = hand_features(X_train.tolist())
test_hand  = hand_features(X_test.tolist())
print('Shapes:', train_hand.shape, test_hand.shape)
print('\nFirst 3 rows (length, excl, upper_ratio, polarity):')
print(np.round(train_hand[:3], 3))

In [ ]:
# Quick sanity check -- polarity should correlate with the star label.
polarity_train = train_hand[:, 3]
print('Mean polarity by class (train):')
for cls in [0, 1]:
    m = polarity_train[y_train.values == cls].mean()
    print(f'  class {cls} ({["negative","positive"][cls]}):  {m:+.3f}')

In [ ]:
# Stack TF-IDF (sparse) with hand-crafted (dense) using scipy.sparse.hstack.
# Converting the dense block to CSR keeps everything sparse-efficient.
X_train_combo = sp.hstack([X_train_tfidf, sp.csr_matrix(train_hand)]).tocsr()
X_test_combo  = sp.hstack([X_test_tfidf,  sp.csr_matrix(test_hand)]).tocsr()

print(f'Combined train shape: {X_train_combo.shape}')

# Logistic Regression on the combined matrix -- LR handles the mix of scales fine.
lr_combo = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_combo, y_train)
acc_combo = accuracy_score(y_test, lr_combo.predict(X_test_combo))
print(f'\nLogReg + TF-IDF only     : {accuracy_score(y_test, predictions["LogisticRegression"]):.4f}')
print(f'LogReg + TF-IDF + hand-crafted: {acc_combo:.4f}')

### Lab 6.1 -- Engineer Two More Features

**Your tasks:**
1. Pick two more features you think might help. Ideas:
   - Average word length (longer words -> more educated review -> maybe positive?).
   - Number of `?` marks.
   - Presence of emojis or smileys (`:)` / `:(`).
   - Subjectivity (TextBlob's other score: `TextBlob(t).sentiment.subjectivity`).
2. Write a function `extra_features(texts)` returning an `(n, 2)` array.
3. `hstack` it onto `X_train_combo` / `X_test_combo` and retrain Logistic Regression.
4. Report whether the two extra features moved the needle (they may not -- that's a legitimate finding).


In [ ]:
# Lab 6.1 -- starter code
def extra_features(texts):
    feats = np.zeros((len(texts), 2), dtype=np.float64)
    for i, t in enumerate(texts):
        # Feature A
        feats[i, 0] = None  # YOUR CODE
        # Feature B
        feats[i, 1] = None  # YOUR CODE
    return feats

# Build combined matrices (only runs if you implemented extra_features)
try:
    train_extra = extra_features(X_train.tolist())
    test_extra  = extra_features(X_test.tolist())

    X_train_full = sp.hstack([X_train_combo, sp.csr_matrix(train_extra)]).tocsr()
    X_test_full  = sp.hstack([X_test_combo,  sp.csr_matrix(test_extra)]).tocsr()

    lr_full = None  # YOUR CODE -- fit LogisticRegression on (X_train_full, y_train)
    acc_full = None  # YOUR CODE -- test accuracy

    if acc_full is not None:
        print(f'LogReg + TF-IDF + 4 hand + 2 extra: {acc_full:.4f}')
        print(f'Delta vs 4-hand version           : {acc_full - acc_combo:+.4f}')
except TypeError:
    print('Implement the feature functions first!')

## Section 7: Final Lab -- SMS Spam Detection

Time to prove the pipeline generalizes. The **SMS Spam Collection** is a classic dataset: ~5,500 English SMS messages labeled `ham` or `spam`.

Dataset: https://www.dropbox.com/scl/fi/yy0b8tblxx787vw0ncbm4/SMSSpamCollection.tsv?rlkey=iuk84q9leb2hcyisuvqe4c1hn&dl=1

Columns: `label` (`ham`/`spam`), `message` (raw text). Tab-separated.

### Your mission
Build an end-to-end spam classifier. Deliverables:
1. **Load** the TSV from the URL above.
2. **Map** labels: `ham` -> 0, `spam` -> 1. `train_test_split` with `stratify=y`, `test_size=0.2`.
3. **Baseline.** Majority-class accuracy -- print it. (Expect ~87% because most SMS are ham.)
4. **Vectorize.** `TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words='english')`.
5. **Train three classifiers**: MultinomialNB, RandomForest, LogisticRegression.
6. **Pick the winner** by macro-F1 (precision/recall matter more than accuracy on imbalanced data!).
7. **Plot** a confusion matrix for the winner. Which errors are more costly -- missed spam (FN) or flagged ham (FP)?

This is homework. Use the code above as a template.


In [ ]:
# Final lab -- SMS spam. Starter code.
SMS_URL = 'https://www.dropbox.com/scl/fi/yy0b8tblxx787vw0ncbm4/SMSSpamCollection.tsv?rlkey=iuk84q9leb2hcyisuvqe4c1hn&dl=1'

# 1. Load the TSV (hint: pd.read_csv(SMS_URL, sep='\t', names=['label', 'message']))
sms = None  # YOUR CODE

# 2. Map labels and split
X_sms = None  # YOUR CODE -- the message column
y_sms = None  # YOUR CODE -- map ham=0, spam=1

Xtr_sms, Xte_sms, ytr_sms, yte_sms = (None, None, None, None)  # YOUR CODE -- train_test_split(...)

# 3. Majority-class baseline
base_sms = None  # YOUR CODE
if base_sms is not None:
    print(f'SMS majority-class baseline: {base_sms:.4f}')

# 4. Vectorize
sms_vect = None  # YOUR CODE
Xtr_vec = None  # YOUR CODE
Xte_vec = None  # YOUR CODE

# 5. Train all three
sms_models = {
    'MultinomialNB'      : None,  # YOUR CODE
    'RandomForest'       : None,  # YOUR CODE
    'LogisticRegression' : None,  # YOUR CODE
}

# 6. Pick winner by macro-F1 -- print classification_report for each
# YOUR CODE

# 7. Confusion matrix for the winner (with seaborn heatmap)
# YOUR CODE

## Wrap-up & Next Steps

You just built, from a blank notebook, a complete classical text-classification pipeline:

1. **Frame the problem** as binary classification with a clean train/test split.
2. **Vectorize** with `CountVectorizer` (BoW) or `TfidfVectorizer`, tuning `ngram_range`, `stop_words`, `max_features`, `min_df`.
3. **Beat the majority baseline** with `MultinomialNB`, `RandomForestClassifier`, or `LogisticRegression`.
4. **Read the metrics** -- precision, recall, F1 -- and pick the right one for the business problem.
5. **Engineer hand-crafted features** and stack them onto the sparse matrix with `scipy.sparse.hstack`.
6. **Ship the same pipeline** on a totally different dataset (SMS spam) with five lines of change.

### Self-check quiz
1. You have 100 tweets, each ~15 tokens. Do you prefer unigrams or unigrams+bigrams? Why?
2. In TF-IDF, what happens to a word that appears in every document?
3. Your NB model hits 98% accuracy but the dataset is 99% one class. Is the model good?

### Common pitfalls to remember
- Never call `fit_transform` on the test set -- always `transform`.
- Don't call `.toarray()` on a full 20K-feature matrix unless you enjoy OOM errors.
- Default English stop words drop `not` -- bad for sentiment. Curate the list.
- Majority-class baseline first. Always.

### Up next: Notebook 3
We will keep the classical toolkit but add: logistic regression deep-dive, metrics mastery (precision-recall curves, ROC), gradient boosting, and a pivot to unsupervised clustering (K-Means, DBSCAN, dendrograms).
